In [1]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, SVR
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, \
RocCurveDisplay, roc_auc_score, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import log_loss, f1_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, BaggingClassifier, BaggingRegressor, RandomForestClassifier, RandomForestRegressor, StackingClassifier, StackingRegressor
from sklearn.linear_model import ridge_regression, ElasticNet
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, cross_val_score, KFold, StratifiedKFold

import xgboost as xgb
import lightgbm as lgb

import catboost

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/dbda2025cdac-maker/Machine-Learning/refs/heads/main/datasets/pizza.csv')
df

,Promote,Sales
0,23,554
1,56,1339
2,34,815
3,25,609
4,67,1600
5,82,2000
6,46,1000
7,14,300
8,6,150
9,47,1150


In [4]:
x = df[['Promote']]
y = df['Sales']

In [5]:
lr = LinearRegression()
kfold = KFold(n_splits=5, shuffle=True, random_state=25)
next(iter(kfold.split(df)))

(array([ 0,  1,  2,  3,  4,  5,  6,  8,  9, 11, 12, 13, 15, 16, 18]),
 array([ 7, 10, 14, 17]))

In [6]:
next(kfold.split(df))
next(kfold.split(df))

(array([ 0,  1,  2,  3,  4,  5,  6,  8,  9, 11, 12, 13, 15, 16, 18]),
 array([ 7, 10, 14, 17]))

# Method - 1

In [7]:
scores = []

for i , (train_index, test_index) in enumerate(kfold.split(df)):
    print(f"Fold {i}: ")
    print(f" Train index = {train_index} ")
    print(f" Test index = {test_index} ")
    
    x_train, y_train = x.iloc[train_index], y.iloc[train_index]
    x_test, y_test = x.iloc[test_index], y.iloc[test_index]
    lr.fit(x_train, y_train)
    y_pred = lr.predict(x_test)
    scores.append([r2_score(y_test, y_pred)])
print(scores)
print("CV Scores = ", np.mean(scores))

Fold 0: 
 Train index = [ 0  1  2  3  4  5  6  8  9 11 12 13 15 16 18] 
 Test index = [ 7 10 14 17] 
Fold 1: 
 Train index = [ 2  4  5  7  8  9 10 11 12 13 14 15 16 17 18] 
 Test index = [0 1 3 6] 
Fold 2: 
 Train index = [ 0  1  2  3  4  6  7  8 10 12 13 14 15 16 17] 
 Test index = [ 5  9 11 18] 
Fold 3: 
 Train index = [ 0  1  3  4  5  6  7  9 10 11 12 14 15 17 18] 
 Test index = [ 2  8 13 16] 
Fold 4: 
 Train index = [ 0  1  2  3  5  6  7  8  9 10 11 13 14 16 17 18] 
 Test index = [ 4 12 15] 
[[0.977595082312369], [0.9790917952286047], [0.9910691519887425], [0.9833261216932678], [0.9713879680534088]]
CV Scores =  0.9804940238552785


# Method - 2

In [8]:
results = cross_val_score(lr,x,y, scoring='r2', cv = kfold)
print("Scores: ", results)
print('CV Score: ', np.mean(results))

Scores:  [0.97759508 0.9790918  0.99106915 0.98332612 0.97138797]
CV Score:  0.9804940238552785


In [9]:
from sklearn.model_selection import cross_val_score, KFold

In [10]:
glass = pd.read_csv('https://raw.githubusercontent.com/dbda2025cdac-maker/Machine-Learning/refs/heads/main/Cases/Glass_Identification/Glass.csv')
x,y = glass.drop('Type', axis = 1), glass['Type']
le = LabelEncoder()
y = le.fit_transform(y)
lr = LogisticRegression()
kfold = KFold(n_splits=5, shuffle=True, random_state=25)
results = cross_val_score(lr, x,y, scoring='accuracy', cv = kfold)
results

array([0.65116279, 0.58139535, 0.69767442, 0.51162791, 0.5952381 ])

In [11]:
np.mean(results)

np.float64(0.6074197120708749)

In [12]:
glass = pd.read_csv('https://raw.githubusercontent.com/dbda2025cdac-maker/Machine-Learning/refs/heads/main/Cases/Glass_Identification/Glass.csv')
x,y = glass.drop('Type', axis = 1), glass['Type']
le = LabelEncoder()
y = le.fit_transform(y)
lr = LogisticRegression()
kfold = KFold(n_splits=5, shuffle=True, random_state=25)
results = cross_val_score(lr, x,y, scoring='f1_macro', cv = kfold)
results

array([0.52884615, 0.38561254, 0.45481181, 0.33769063, 0.52142857])

In [13]:
np.mean(results)

np.float64(0.44567794049538473)

In [14]:
solvers = ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
penalties = ['l2', None]
_c = np.linspace(0.001, 15, 20)
scores = []
kfold = KFold(n_splits=5, shuffle=True, random_state=25)
for s in tqdm(solvers):
    for p in penalties:
        for _ in _c:
            try:
                lr = LogisticRegression(solver=s, penalty=p, max_iter=500, C=_)

                # using cross val score
                results = cross_val_score(lr, x,y, scoring='accuracy', cv = kfold)

                scores.append([_,s,p, np.mean(results)])
            except:
                pass
df_score = pd.DataFrame(scores, columns=['c', 'Solver', 'Penalty', 'Accuracy score'])
df_score.sort_values('Accuracy score', ascending=False)

100%|██████████| 5/5 [01:36<00:00, 19.22s/it]


,c,Solver,Penalty,Accuracy score
112,9.474053,newton-cholesky,None,0.635105
113,10.263474,newton-cholesky,None,0.635105
114,11.052895,newton-cholesky,None,0.635105
115,11.842316,newton-cholesky,None,0.635105
100,0.001000,newton-cholesky,None,0.635105
...,...,...,...,...
120,0.001000,sag,l2,0.331783
160,0.001000,saga,l2,0.331783
0,0.001000,lbfgs,l2,0.317497
40,0.001000,newton-cg,l2,0.317497


In [15]:
depths = [None, 3,4,5,6,7]
min_samples = [2,10,0.025,0.01,0.05,0.1]
min_leaf = [1,10,0.025,0.01,0.05,0.1]
scores = []

kfold = KFold(n_splits=5, shuffle = True, random_state=25)

for d in tqdm(depths):
    for ms in min_samples:
        for ml in min_leaf:
            dtr = DecisionTreeClassifier(random_state=25, max_depth=d, min_samples_split=ms, min_samples_leaf=ml)
            # dtr.fit(x_train, y_train)
            results = cross_val_score(dtr, x,y,scoring = 'f1_macro')
            # y_pred = dtr.predict(x_test)
            scores.append([d,ms,ml,np.mean(results)])

df_score = pd.DataFrame(scores, columns=['depth', 'min_sample_split', 'min_sample_leaf', 'score'])
df_score.sort_values('score', ascending=False)

100%|██████████| 6/6 [00:06<00:00,  1.11s/it]


,depth,min_sample_split,min_sample_leaf,score
140,5.0,0.100,0.025,0.545244
32,NaN,0.100,0.025,0.543224
141,5.0,0.100,0.010,0.543217
138,5.0,0.100,1.000,0.542567
72,4.0,2.000,1.000,0.541371
...,...,...,...,...
59,3.0,0.010,0.100,0.424319
47,3.0,10.000,0.100,0.424319
41,3.0,2.000,0.100,0.424319
71,3.0,0.100,0.100,0.424319


In [16]:
# StratifiedKFold
depths = [None, 3,4,5,6,7]
min_samples = [2,10,0.025,0.01,0.05,0.1]
min_leaf = [1,10,0.025,0.01,0.05,0.1]
scores = []

kfold = StratifiedKFold(n_splits=5, shuffle = True, random_state=25)

for d in tqdm(depths):
    for ms in min_samples:
        for ml in min_leaf:
            dtr = DecisionTreeClassifier(random_state=25, max_depth=d, min_samples_split=ms, min_samples_leaf=ml)
            # dtr.fit(x_train, y_train)
            results = cross_val_score(dtr, x,y,scoring = 'f1_macro')
            # y_pred = dtr.predict(x_test)
            scores.append([d,ms,ml,np.mean(results)])

df_score = pd.DataFrame(scores, columns=['depth', 'min_sample_split', 'min_sample_leaf', 'score'])
df_score.sort_values('score', ascending=False)

100%|██████████| 6/6 [00:06<00:00,  1.15s/it]


,depth,min_sample_split,min_sample_leaf,score
140,5.0,0.100,0.025,0.545244
32,NaN,0.100,0.025,0.543224
141,5.0,0.100,0.010,0.543217
138,5.0,0.100,1.000,0.542567
72,4.0,2.000,1.000,0.541371
...,...,...,...,...
59,3.0,0.010,0.100,0.424319
47,3.0,10.000,0.100,0.424319
41,3.0,2.000,0.100,0.424319
71,3.0,0.100,0.100,0.424319
